# 代码文件1：图片难度分层分析

本notebook用于分析所有样本的gemini描述token数和xml token数，确定图片难度分层标准，创建图片难度分层数据表单。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from scipy import stats

# 表格美化显示
try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False
    print("提示: 安装 tabulate 可以获得更好的表格显示效果: pip install tabulate")

try:
    from rich.console import Console
    from rich.table import Table
    from rich import box
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    print("提示: 安装 rich 可以获得彩色表格显示效果: pip install rich")

# 设置中文字体（如果需要）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 优化pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)  # 不限制列宽，让内容完整显示
pd.set_option('display.precision', 3)
pd.set_option('display.float_format', lambda x: f'{x:.3f}' if pd.notna(x) else '')
pd.set_option('display.expand_frame_repr', False)  # 不换行显示DataFrame

# 定义美化表格显示函数
def display_table(df, title="", max_rows=20, use_rich=True):
    """美化显示DataFrame表格"""
    if title:
        print(f"\n{'='*80}")
        print(f"{title:^80}")
        print(f"{'='*80}")
    
    # 优先使用rich显示（如果可用且数据不太大）
    if HAS_RICH and use_rich and len(df) <= 50 and len(df.columns) <= 15:
        console = Console(width=None)  # 不限制控制台宽度
        table = Table(show_header=True, header_style="bold magenta", box=box.ROUNDED, expand=True)
        
        # 添加列（不换行，允许更宽的列）
        for col in df.columns:
            table.add_column(str(col), style="cyan", no_wrap=True)
        
        # 添加行
        for idx, row in df.head(max_rows).iterrows():
            table.add_row(*[str(val) if pd.notna(val) else 'N/A' for val in row])
        
        console.print(table)
        if len(df) > max_rows:
            print(f"\n... (还有 {len(df) - max_rows} 行未显示)")
    
    # 使用tabulate显示
    elif HAS_TABULATE:
        # 临时设置pandas选项确保列内容完整
        old_max_colwidth = pd.get_option('display.max_colwidth')
        pd.set_option('display.max_colwidth', None)
        
        try:
            if len(df) > max_rows:
                print(f"\n显示前{max_rows}行（共{len(df)}行）:")
                print(tabulate(df.head(max_rows), headers='keys', tablefmt='grid', 
                              showindex=False, floatfmt='.3f', numalign='right'))
                print(f"\n... (还有 {len(df) - max_rows} 行)")
            else:
                print(tabulate(df, headers='keys', tablefmt='grid', 
                              showindex=False, floatfmt='.3f', numalign='right'))
        finally:
            pd.set_option('display.max_colwidth', old_max_colwidth)
    
    # 回退到pandas默认显示
    else:
        if len(df) > max_rows:
            print(f"\n显示前{max_rows}行（共{len(df)}行）:")
            print(df.head(max_rows).to_string(max_colwidth=None))
            print(f"\n... (还有 {len(df) - max_rows} 行)")
        else:
            print(df.to_string(max_colwidth=None))
    
    print(f"\n形状: {df.shape[0]} 行 × {df.shape[1]} 列")

# 数据根目录
# Resolve VCG-Bench root from the current notebook working directory.
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'src').exists():
            return path
    return start

BASE_DIR = find_repo_root()
DATA_DIR = BASE_DIR / 'data'
ANALYSIS_DIR = DATA_DIR / 'analysis'

# 创建必要的目录结构
(ANALYSIS_DIR / 'difficulty_stratification' / 'tables').mkdir(parents=True, exist_ok=True)
(ANALYSIS_DIR / 'difficulty_stratification' / 'figures').mkdir(parents=True, exist_ok=True)

print(f"数据根目录: {BASE_DIR}")
print(f"分析结果目录: {ANALYSIS_DIR}")


In [ ]:
# 读取所有样本的gemini描述token数和xml token数
# 从task1_benchmark目录读取performance.json文件

task1_benchmark_dir = DATA_DIR / 'task1_benchmark'
data_list = []

# 遍历所有domain目录
for domain_dir in task1_benchmark_dir.iterdir():
    if not domain_dir.is_dir() or domain_dir.name.startswith('.'):
        continue
    
    domain = domain_dir.name
    
    # 遍历每个sample目录
    for sample_dir in domain_dir.iterdir():
        if not sample_dir.is_dir() or not sample_dir.name.startswith('sample_'):
            continue
        
        sample_id = sample_dir.name
        
        # 查找gemini-3-pro-preview的performance.json文件
        performance_file = sample_dir / 'model_gemini-3-pro-preview' / 'performance.json'
        
        if performance_file.exists():
            try:
                with open(performance_file, 'r', encoding='utf-8') as f:
                    perf_data = json.load(f)
                
                # 提取token信息
                token_usage = perf_data.get('token_usage', {})
                description_tokens = token_usage.get('description', {}).get('completion_tokens', 0)
                xml_tokens = token_usage.get('xml', {}).get('completion_tokens', 0)
                
                data_list.append({
                    'sample_id': sample_id,
                    'domain': domain,
                    'gemini_description_tokens': description_tokens,
                    'xml_tokens': xml_tokens
                })
            except Exception as e:
                print(f"Error reading {performance_file}: {e}")

# 构建DataFrame
df = pd.DataFrame(data_list)

print(f"总样本数: {len(df)}")
display_table(df.head(10), title="前10条数据预览")

print(f"\n数据统计概览:")
display_table(df.describe().T, title="数据统计信息")


In [ ]:
# 生成Token统计摘要表格
# 对所有样本计算统计量，并按domain分组计算

stats_list = []

# 定义要计算的统计量
statistics = ['mean', 'median', 'std', 'min', 'max', 'q25', 'q50', 'q75']

# 对所有样本计算统计量
for metric in ['gemini_description_tokens', 'xml_tokens']:
    for stat in statistics:
        if stat == 'q25':
            value = df[metric].quantile(0.25)
        elif stat == 'q50':
            value = df[metric].quantile(0.50)
        elif stat == 'q75':
            value = df[metric].quantile(0.75)
        else:
            value = getattr(df[metric], stat)()
        
        row = {
            'metric': metric,
            'statistic': stat,
            'all_samples': value
        }
        
        # 按domain分组计算
        for domain in df['domain'].unique():
            domain_data = df[df['domain'] == domain][metric]
            if stat == 'q25':
                domain_value = domain_data.quantile(0.25)
            elif stat == 'q50':
                domain_value = domain_data.quantile(0.50)
            elif stat == 'q75':
                domain_value = domain_data.quantile(0.75)
            else:
                domain_value = getattr(domain_data, stat)()
            
            row[domain] = domain_value
        
        stats_list.append(row)

# 构建统计表格
stats_df = pd.DataFrame(stats_list)

# 保存为CSV
output_path = ANALYSIS_DIR / 'difficulty_stratification' / 'tables' / 'token_statistics_summary.csv'
stats_df.to_csv(output_path, index=False, encoding='utf-8')

display_table(stats_df, title="Token统计摘要")
print(f"\n已保存到: {output_path}")


In [ ]:
# 绘制Token数分布图（图1）
# 创建2x1子图布局，展示gemini_description_tokens和xml_tokens的分布

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# 子图1: gemini_description_tokens的直方图 + KDE曲线
ax1 = axes[0]
ax1.hist(df['gemini_description_tokens'], bins=50, density=True, alpha=0.7, label='Histogram', color='skyblue')
df['gemini_description_tokens'].plot.density(ax=ax1, label='KDE', color='darkblue', linewidth=2)

# 标注统计线
mean_val = df['gemini_description_tokens'].mean()
median_val = df['gemini_description_tokens'].median()
q25_val = df['gemini_description_tokens'].quantile(0.25)
q75_val = df['gemini_description_tokens'].quantile(0.75)

ax1.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.0f}')
ax1.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.0f}')
ax1.axvline(q25_val, color='orange', linestyle=':', linewidth=1.5, label=f'Q25: {q25_val:.0f}')
ax1.axvline(q75_val, color='orange', linestyle=':', linewidth=1.5, label=f'Q75: {q75_val:.0f}')

ax1.set_xlabel('Gemini Description Tokens', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Distribution of Gemini Description Tokens', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 子图2: xml_tokens的直方图 + KDE曲线
ax2 = axes[1]
ax2.hist(df['xml_tokens'], bins=50, density=True, alpha=0.7, label='Histogram', color='lightcoral')
df['xml_tokens'].plot.density(ax=ax2, label='KDE', color='darkred', linewidth=2)

# 标注统计线
mean_val = df['xml_tokens'].mean()
median_val = df['xml_tokens'].median()
q25_val = df['xml_tokens'].quantile(0.25)
q75_val = df['xml_tokens'].quantile(0.75)

ax2.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.0f}')
ax2.axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.0f}')
ax2.axvline(q25_val, color='orange', linestyle=':', linewidth=1.5, label=f'Q25: {q25_val:.0f}')
ax2.axvline(q75_val, color='orange', linestyle=':', linewidth=1.5, label=f'Q75: {q75_val:.0f}')

ax2.set_xlabel('XML Tokens', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Distribution of XML Tokens', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'difficulty_stratification' / 'figures' / 'token_distribution_all_samples.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 绘制按领域Token数分布图（图2）
# 使用箱线图展示不同领域的XML token数分布

plt.figure(figsize=(14, 8))

# 准备数据：按domain分组
domain_data = [df[df['domain'] == domain]['xml_tokens'].values for domain in sorted(df['domain'].unique())]
domain_labels = [domain.replace('domain_', '').replace('_', ' ').title() for domain in sorted(df['domain'].unique())]

# 绘制箱线图
bp = plt.boxplot(domain_data, labels=domain_labels, patch_artist=True, 
                 showmeans=True, meanline=True)

# 美化箱线图
colors = plt.cm.Set3(np.linspace(0, 1, len(bp['boxes'])))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

plt.xticks(rotation=45, ha='right')
plt.ylabel('XML Tokens', fontsize=12)
plt.xlabel('Domain', fontsize=12)
plt.title('XML Token Distribution by Domain', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

# 保存图片
output_path = ANALYSIS_DIR / 'difficulty_stratification' / 'figures' / 'token_distribution_by_domain.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"已保存图片到: {output_path}")

plt.show()


In [ ]:
# 确定难度分层标准
# 根据xml_tokens的分布和分位数确定阈值

# 计算分位数
q25 = df['xml_tokens'].quantile(0.25)
q50 = df['xml_tokens'].quantile(0.50)
q75 = df['xml_tokens'].quantile(0.75)

print("XML Tokens统计信息:")
print(f"  25%分位数 (Q25): {q25:.0f}")
print(f"  50%分位数 (Q50/中位数): {q50:.0f}")
print(f"  75%分位数 (Q75): {q75:.0f}")
print(f"  均值: {df['xml_tokens'].mean():.0f}")
print(f"  标准差: {df['xml_tokens'].std():.0f}")
print(f"  最小值: {df['xml_tokens'].min():.0f}")
print(f"  最大值: {df['xml_tokens'].max():.0f}")

# 根据分位数确定阈值（可以根据实际情况调整）
# 方案1: 使用三分位数
# Easy: < Q33, Medium: Q33-Q66, Hard: >= Q66
q33 = df['xml_tokens'].quantile(0.33)
q66 = df['xml_tokens'].quantile(0.66)

# 方案2: 使用固定阈值（根据观察调整）
# 可以根据实际分布情况选择方案

# 这里使用方案1，但也可以手动设置阈值
# THRESHOLD_EASY = q33  # 可以改为固定值，如1000
# THRESHOLD_MEDIUM = q66  # 可以改为固定值，如2000
THRESHOLD_EASY = q33  # 可以改为固定值，如1000
THRESHOLD_MEDIUM = 14000  # 可以改为固定值，如2000

print(f"\n建议的分层阈值（基于分位数）:")
print(f"  Easy: xml_tokens < {THRESHOLD_EASY:.0f}")
print(f"  Medium: {THRESHOLD_EASY:.0f} <= xml_tokens < {THRESHOLD_MEDIUM:.0f}")
print(f"  Hard: xml_tokens >= {THRESHOLD_MEDIUM:.0f}")

# 或者使用固定阈值（根据实际需求调整）
# THRESHOLD_EASY = 1000
# THRESHOLD_MEDIUM = 2000

# 预览分层结果
df['preview_difficulty'] = df['xml_tokens'].apply(
    lambda x: 'Easy' if x < THRESHOLD_EASY else ('Medium' if x < THRESHOLD_MEDIUM else 'Hard')
)

print(f"\n预览分层结果:")
print(df['preview_difficulty'].value_counts())
print(f"\n各难度等级占比:")
print(df['preview_difficulty'].value_counts(normalize=True) * 100)


In [ ]:
# 创建图片难度分层表单
# 根据确定的阈值，为每个样本分配难度等级

# 使用上面确定的阈值（可以根据实际情况修改）
# 如果使用固定阈值，取消下面的注释并修改
# THRESHOLD_EASY = 1000
# THRESHOLD_MEDIUM = 2000

def assign_difficulty(xml_tokens):
    """根据xml_tokens分配难度等级"""
    if xml_tokens < THRESHOLD_EASY:
        return 'Easy', f'xml_tokens < {THRESHOLD_EASY:.0f}'
    elif xml_tokens < THRESHOLD_MEDIUM:
        return 'Medium', f'{THRESHOLD_EASY:.0f} <= xml_tokens < {THRESHOLD_MEDIUM:.0f}'
    else:
        return 'Hard', f'xml_tokens >= {THRESHOLD_MEDIUM:.0f}'

# 为每个样本分配难度等级
df['difficulty_level'] = df['xml_tokens'].apply(lambda x: assign_difficulty(x)[0])
df['difficulty_criteria'] = df['xml_tokens'].apply(lambda x: assign_difficulty(x)[1])

# 构建最终表格
difficulty_df = df[['sample_id', 'domain', 'gemini_description_tokens', 'xml_tokens', 
                    'difficulty_level', 'difficulty_criteria']].copy()

# 保存为CSV
output_path = ANALYSIS_DIR / 'difficulty_stratification' / 'tables' / 'image_difficulty_levels.csv'
difficulty_df.to_csv(output_path, index=False, encoding='utf-8')

print("图片难度分层数据:")
print(f"总样本数: {len(difficulty_df)}")

# 显示难度等级分布
difficulty_counts = difficulty_df['difficulty_level'].value_counts()
difficulty_pct = difficulty_df['difficulty_level'].value_counts(normalize=True) * 100
difficulty_dist = pd.DataFrame({
    '数量': difficulty_counts,
    '占比(%)': difficulty_pct.round(2)
})
display_table(difficulty_dist, title="难度等级分布")

# 显示按领域分布
domain_difficulty = difficulty_df.groupby(['domain', 'difficulty_level']).size().unstack(fill_value=0)
domain_difficulty.index = [d.replace('domain_', '').replace('_', ' ').title() for d in domain_difficulty.index]
display_table(domain_difficulty, title="按领域难度分布")

# 显示前10条数据
display_table(difficulty_df.head(10), title="前10条数据预览")
print(f"\n已保存到: {output_path}")


In [ ]:
# 统一设置难度顺序
difficulty_order = ['Easy', 'Medium', 'Hard']
# 统一设置颜色映射：Easy(绿), Medium(黄), Hard(红)
colors_map = ['#90EE90', '#FFD700', '#FF6347']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 子图1: 堆叠柱状图 (按 Domain) ---
ax1 = axes[0]

# 分组计数并重塑表格
difficulty_by_domain = difficulty_df.groupby(['domain', 'difficulty_level']).size().unstack(fill_value=0)

# 核心步骤：重新排序列，确保堆叠顺序从下到上是 Easy -> Medium -> Hard
# 同时也保证了颜色能正确对应
difficulty_by_domain = difficulty_by_domain.reindex(columns=difficulty_order, fill_value=0)

# 格式化 Domain 名称
difficulty_by_domain.index = [d.replace('domain_', '').replace('_', ' ').title() for d in difficulty_by_domain.index]

# 绘图
difficulty_by_domain.plot(kind='bar', stacked=True, ax=ax1, 
                          color=colors_map, alpha=0.8)

ax1.set_xlabel('Domain', fontsize=12)
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.set_title('Difficulty Distribution by Domain', fontsize=14, fontweight='bold')
ax1.legend(title='Difficulty Level', loc='upper right')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True, alpha=0.3, axis='y')

# --- 子图2: 饼图 (整体分布) ---
ax2 = axes[1]

# 确保饼图的数据顺序也遵循 Easy -> Medium -> Hard，使颜色对应一致
difficulty_counts = difficulty_df['difficulty_level'].value_counts().reindex(difficulty_order)

ax2.pie(difficulty_counts.values, 
        labels=difficulty_counts.index, 
        autopct='%1.1f%%',
        colors=colors_map, 
        startangle=90, 
        textprops={'fontsize': 12})
ax2.set_title('Overall Difficulty Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()

# 保存图片 (请确保 ANALYSIS_DIR 已定义，或修改为直接保存的文件名)
try:
    output_path = ANALYSIS_DIR / 'difficulty_stratification' / 'figures' / 'difficulty_distribution.png'
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"已保存图片到: {output_path}")
except NameError:
    plt.savefig('difficulty_distribution.png', dpi=300, bbox_inches='tight')
    print("已保存图片到当前目录: difficulty_distribution.png")

plt.show()